In [ ]:
"""
# Pilot Analysis Notebook

This notebook analyzes pilot period data compared to baseline data.

**Workflow Prerequisites:**
1. Run `4 aggregate.ipynb` for both baseline and pilot folders
2. Ensure both baseline and pilot summaries are generated

**Key Outputs:**
- Comparison of baseline vs pilot periods across all GBD categories
- Plant-based vs animal product shifts
- Statistical significance tests for changes
- Visualizations of trends over time
"""

from dotenv import load_dotenv
from gbd_foodservice_insights.categories import get_food_categories
from gbd_foodservice_insights.plotting_utils import setup_gbd_fonts
from gbd_foodservice_insights.report.aggregation import identify_overall_drivers
from gbd_foodservice_insights.report.plots import plot_overall_drivers
from gbd_foodservice_insights_lab.notebook_runscript_setup import (
    load_client_metadata,
    setup_pandas_display,
)
from gbd_foodservice_insights_lab.pilot.analysis import (
    analyze_animal_product_consumption,
    analyze_category_consumption,
    analyze_meat_consumption,
    analyze_milk_consumption,
    calculate_plant_milk_percentage,
    calculate_product_overlap,
    compare_plant_based_product_counts,
    get_category_baseline_pilot_comparison,
    load_all_pilot_data,
    plant_animal_split,
)
from gbd_foodservice_insights_lab.pilot.plots import (
    plot_category_trends,
    plot_diner_meal_numbers_over_time,
    plot_kilos_over_time,
    plot_kilos_per_diner_meal_pilot_changes_by_category,
    plot_kilos_per_diner_over_time,
    plot_milk_split,
    plot_plant_animal_split,
    plot_unique_products_by_category,
    plot_unique_products_over_time,
    print_baseline_pilot_percent_changes,
    print_plant_based_product_changes,
    print_product_overlap_summary,
)

setup_pandas_display()
setup_gbd_fonts()

# Load metadata saved from notebook 1
config, ENV_PATH, PDF_EXTRACTED = load_client_metadata(step="pilot")
load_dotenv(dotenv_path=ENV_PATH)

client = config["client"]
analysis_context = config["analysis_context"]
procurement_serving = config["procurement_serving"]
sub_client_name = config["sub_client_name"]
baseline_input_file = config["baseline_input_file"]
pilot_input_file = config["pilot_input_file"]

# 1. Build datasets

In [ ]:
# Load all pilot analysis data in one call
data = load_all_pilot_data(baseline_input_file, pilot_input_file, include_total_meat=True)

# Unpack for convenience
template_data = data["template_data"]
monthly_product_data = data["monthly_product_data"]
monthly_category_data = data["monthly_category_data"]
diner_meal_data = data["diner_meal_data"]
monthly_product_category_data = data["monthly_product_category_data"]
period_category_data = data["period_category_data"]

print(f"Loaded {len(data)} datasets:")
for name, df in data.items():
    print(f"  - {name}: {df.shape[0]} rows, {df.shape[1]} columns")

# 2. validate the combined baseline and pilot data to see if there are any worrying patterns

## Topline baseline vs pilot changes


In [ ]:
print_baseline_pilot_percent_changes(
    monthly_category_data=monthly_category_data,
    diner_meal_data=diner_meal_data,
    monthly_product_data=monthly_product_data,
)

## Product name overlap between baseline and pilot

In [ ]:
product_overlap = calculate_product_overlap(monthly_product_data)
print_product_overlap_summary(product_overlap)

## Check that diner-meal numbers have remained mostly constant month to month through baseline and pilot

In [ ]:
# Another way to plot this data would be to overlay baseline and pilot and have just
# month on the X axis.
# While this has the advantage of being able to spot seasonality in the data,
# it has the disadvantage that if the pilot runs over multiple years, for example,
# December, January, February,
# then it can be very confusing how to order it.
# January, February and December, is misleading
# December, January, February is correct but not intuitive.

diner_numbers_plot = plot_diner_meal_numbers_over_time(diner_meal_data)

# TODO retrofit this code:
# add_plot(
#     baseline_report,
#     n_products_per_month_plot,
#     "Figure 2: Number of unique products per month",
# )

## Check that the number of unique products has remained mostly constant month to month through baseline and pilot

In [ ]:
unique_products_plot = plot_unique_products_over_time(monthly_product_data)

In [ ]:
unique_products_by_category_plot = plot_unique_products_by_category(monthly_product_category_data)

## Check how the number of unique products changed within the plant-based meta category

In [ ]:
plant_based_product_changes = compare_plant_based_product_counts(monthly_product_category_data)

plant_based_product_changes_display = plant_based_product_changes[
    [
        "meta_category",
        "category",
        "baseline_count",
        "pilot_count",
        "change",
        "pct_change",
        "retained_count",
        "products_added_count",
        "products_removed_count",
        "increased",
    ]
]

plant_based_product_changes_display

In [ ]:
print_plant_based_product_changes(plant_based_product_changes)

In [ ]:
food_only_categories = get_food_categories(lowercase=True)

monthly_category_data_food_only = monthly_category_data.loc[
    monthly_category_data["category"].isin(food_only_categories), :
]

total_food_over_time_plot = plot_kilos_over_time(
    monthly_category_data_food_only, per_diner_meal=False, compare_baseline_pilot=True
)
# add_plot(
#     baseline_report,
#     total_food_over_time_plot,
#     "Figure 3: Total Food (not including milks) Over Time",
# )

In [ ]:
kilos_per_diner_plot = plot_kilos_per_diner_over_time(
    monthly_category_data_food_only, diner_meal_data
)

# Compare baseline and pilot kilos per diner-meal for each category

In [ ]:
plot = plot_kilos_per_diner_meal_pilot_changes_by_category(
    period_category_data=period_category_data
)

In [ ]:
get_category_baseline_pilot_comparison(period_category_data=period_category_data)

## Dig into a few specific categories

In [ ]:
animal_products = [
    "beef and buffalo meat",
    "pork (pig meat)",
    "poultry (chicken & turkey)",
    "fish & mollusks",
    "shellfish (shrimp & lobster)",
    "liquid eggs",
    "shelled eggs",
]

animal_product_consumption = plot_kilos_per_diner_meal_pilot_changes_by_category(
    period_category_data=period_category_data.loc[
        period_category_data["category"].isin(animal_products)
    ],
    figsize=(10, 4.5),
    title=(
        "During the  pilot, procurement of poultry and beef was higher,\n"
        "pork slightly lower, and liquid eggs significantly lower"
    ),
)

## Examine the trajectories of each category more closely 

In [ ]:
monthly_category_plot = plot_category_trends(monthly_category_data, per_diner_meal=True)

# Identify most popular products

In [ ]:
top_n = 5
overall_drivers = identify_overall_drivers(
    monthly_product_data, metric="kilos per diner_meal", top_n=top_n
)
overall_drivers_plot = plot_overall_drivers(overall_drivers, metric="kilos per diner_meal")

# Analyse meat consumption

In [ ]:
meat_per_diner_meal, meat_summary, averted_summary = analyze_meat_consumption(
    period_category_data, diner_meal_data
)
averted_summary

## Category Consumption Analysis

The `analyze_category_consumption()` function can calculate metrics for any combination of categories.

**Simple usage:** Just pass `category_label` and it auto-fetches the categories!  
Supported: `"Meat"`, `"Dairy"`, `"Animal Products"`, `"Eggs"`, `"Plant Protein"`, `"Protein"`, `"Food"`, `"Drink"`

**Advanced usage:** Pass custom `categories` list for specific combinations.

In [ ]:
# Analyze animal products (meat + dairy + eggs)
animal_per_diner_meal, animal_summary, animal_averted = analyze_animal_product_consumption(
    period_category_data, diner_meal_data
)
animal_averted

In [ ]:
# Analyze milk specifically
milk_per_diner_meal, milk_summary, milk_averted = analyze_milk_consumption(
    period_category_data, diner_meal_data
)
milk_averted

In [ ]:
# Simple! Just specify category_label and it auto-fetches the categories
dairy_per_diner_meal, dairy_summary, dairy_averted = analyze_category_consumption(
    period_category_data, diner_meal_data, category_label="Dairy"
)
dairy_averted

In [ ]:
# Example: Analyze eggs with just the label
eggs_per_diner_meal, eggs_summary, eggs_averted = analyze_category_consumption(
    period_category_data, diner_meal_data, category_label="Eggs"
)
print(f"Eggs averted: {eggs_averted.loc[2, 'Value']} kg")

# You can still pass custom categories if you want specific ones
custom_meats = ["beef and buffalo meat", "pork (pig meat)"]
custom_per_diner_meal, custom_summary, custom_averted = analyze_category_consumption(
    period_category_data, diner_meal_data, categories=custom_meats, category_label="Beef & Pork"
)
print(f"Beef & Pork averted: {custom_averted.loc[2, 'Value']} kg")

# Plant-Animal protein split

In [ ]:
plant_percentages = plant_animal_split(monthly_category_data)
plant_animal_split_plot = plot_plant_animal_split(plant_percentages)
plant_percentages

## Zoom in on milk

In [ ]:
pb_milk_percentage = calculate_plant_milk_percentage(period_category_data)

In [ ]:
milk_split_plot = plot_milk_split(pb_milk_percentage)